# SocialBeliefTrace getting started

Public companion notebook: **imports** the installed package, explains the evaluation flow, runs any scenario, and includes a few **basic harness checks** you can re-run after local changes.

## Setup

```bash
cd social-belief-trace
uv sync --group dev
uv run jupyter lab notebooks/getting_started.ipynb
```

Use the project virtualenv as the Jupyter kernel.


In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

from npc_agent_benchmark.harness.policy_registry import REGISTERED_POLICY_MODES
from npc_agent_benchmark.harness.runner import run_episode
from npc_agent_benchmark.models import TraceEvent

# Notebook may be launched with cwd = social-belief-trace/ or social-belief-trace/notebooks/
CWD = Path.cwd()
BB_ROOT = CWD if (CWD / "scenarios").is_dir() else CWD.parent
assert (BB_ROOT / "scenarios").is_dir(), f"scenarios/ not found from {CWD}"
SCENARIOS = BB_ROOT / "scenarios"
OUT_DIR = BB_ROOT / "runs" / "notebook_getting_started"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("root:", BB_ROOT)
print("policy modes:", sorted(REGISTERED_POLICY_MODES))
print("scenarios:", sorted(p.stem for p in SCENARIOS.glob("*.json")))


## Flow (what the harness owns vs what a policy owns)

| Piece | Responsibility |
|-------|----------------|
| Scenario JSON | Agents, needs/trust/beliefs, optional rumors + scheduled messages, config knobs |
| Policy (`BaseAgent.act`) | Choose `Action` (+ optional `ExplanationTrace`) from observation |
| Runner | Message delivery, world effects, witness/decay/reconcile, JSONL `trace_v0`, metrics |
| Metrics (`summarize_v0`) | Trust drift, calibration, source sensitivity, rates, artifact QA |

Observation each tick includes `t`, `needs`, delivered `messages`, and `peer_ids`. Policies must **not** call an LLM for action selection in this benchmark.


## Run any scenario

Set `SCENARIO_ID` to any stem under `scenarios/`, or load your own JSON path.


In [ ]:
def load_scenario(scenario_id_or_path: str | Path) -> dict:
    p = Path(scenario_id_or_path)
    if not p.suffix:
        p = SCENARIOS / f"{scenario_id_or_path}.json"
    return json.loads(p.read_text(encoding="utf-8"))


SCENARIO_ID = "toy_scarcity_v0"
TICKS = 40
SEED = 1
POLICY_MODE = "utility_only"

scenario = load_scenario(SCENARIO_ID)
trace_path = OUT_DIR / f"{scenario.get('id', SCENARIO_ID)}_{POLICY_MODE}_s{SEED}.jsonl"
summary = run_episode(
    scenario=scenario,
    ticks=TICKS,
    seed=SEED,
    trace_path=trace_path,
    policy_mode=POLICY_MODE,
)
print("scenario_id:", summary.get("scenario_id"))
print("avg_final_needs:", summary.get("avg_final_needs"))
print("action_kind_counts:", summary.get("action_kind_counts"))
print("trace:", trace_path.relative_to(BB_ROOT).as_posix())


## Peek at the trace

Validate schema version and skim one event.


In [ ]:
raw_lines = [ln for ln in trace_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
events = [TraceEvent.model_validate_json(ln) for ln in raw_lines]
assert events, "empty trace"
assert all(ev.trace_schema_version == "trace_v0" for ev in events)
ev0 = events[0]
print(f"{len(events)} events; first: t={ev0.t} agent={ev0.agent_id} action={ev0.action.kind}")
print("observation keys:", sorted(ev0.observation.keys()))
print("explanation factors sample:", dict(list(ev0.explanation.factors.items())[:6]))


## Basic tests (notebook-friendly)

These are lightweight checks — not a replacement for `uv run pytest` / `scripts/smoke_test.py`, but useful when exploring.


In [ ]:
def test_trace_schema_and_nonempty(scenario_id: str = "toy_scarcity_v0", ticks: int = 20) -> None:
    sc = load_scenario(scenario_id)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "t.jsonl"
        summary = run_episode(scenario=sc, ticks=ticks, seed=0, trace_path=path, policy_mode="utility_only")
        assert path.exists() and path.stat().st_size > 0
        assert summary.get("scenario_id") == sc.get("id", scenario_id)
        for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            obj = json.loads(line)
            assert obj.get("trace_schema_version") == "trace_v0", f"line {i}"
            assert "action" in obj and "state_after" in obj
    print("PASS test_trace_schema_and_nonempty")


def test_external_stub_is_deterministic_rest(scenario_id: str = "toy_scarcity_v0") -> None:
    sc = load_scenario(scenario_id)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "stub.jsonl"
        summary = run_episode(
            scenario=sc, ticks=15, seed=0, trace_path=path, policy_mode="external_stub"
        )
        kinds = summary.get("action_kind_counts") or {}
        assert set(kinds) == {"rest"}, kinds
        assert summary.get("unique_action_kinds") == 1
    print("PASS test_external_stub_is_deterministic_rest")


def test_provenance_stress_sample_count() -> None:
    # Mirrors scripts/smoke_test.py sample-count expectation when scenario is present.
    sid = "toy_provenance_stress_v0"
    if not (SCENARIOS / f"{sid}.json").exists():
        print("SKIP test_provenance_stress_sample_count (scenario missing)")
        return
    sc = load_scenario(sid)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "p.jsonl"
        summary = run_episode(
            scenario=sc,
            ticks=70,
            seed=0,
            trace_path=path,
            use_provenance_weighting=True,
            enable_rumor_channel=True,
            apply_belief_updates=True,
            policy_mode="utility_only",
        )
        uptake = summary.get("belief_uptake")
        assert isinstance(uptake, dict)
        assert uptake.get("trust_conf_pearson_r_n") == 64, uptake
    print("PASS test_provenance_stress_sample_count")


def test_policy_modes_construct(scenario_id: str = "toy_scarcity_v0") -> None:
    sc = load_scenario(scenario_id)
    # Keep this fast: short ticks, skip legacy mixed.
    for mode in sorted(REGISTERED_POLICY_MODES):
        with tempfile.TemporaryDirectory() as tmp:
            run_episode(
                scenario=sc,
                ticks=5,
                seed=1,
                trace_path=Path(tmp) / f"{mode}.jsonl",
                policy_mode=mode,
            )
    print("PASS test_policy_modes_construct", sorted(REGISTERED_POLICY_MODES))


test_trace_schema_and_nonempty()
test_external_stub_is_deterministic_rest()
test_provenance_stress_sample_count()
test_policy_modes_construct()
print("All notebook basic tests passed.")


## Compare two policy modes on one scenario

Useful for a quick qualitative check before full `scripts/run_all_agg.py` regeneration.


In [ ]:
def quick_compare(scenario_id: str, modes: list[str], *, ticks: int = 40, seed: int = 1) -> None:
    sc = load_scenario(scenario_id)
    rows = []
    for mode in modes:
        with tempfile.TemporaryDirectory() as tmp:
            s = run_episode(
                scenario=sc,
                ticks=ticks,
                seed=seed,
                trace_path=Path(tmp) / "t.jsonl",
                policy_mode=mode,
            )
        rows.append(
            {
                "policy_mode": mode,
                "unique_action_kinds": s.get("unique_action_kinds"),
                "action_kind_counts": s.get("action_kind_counts"),
                "avg_trust_drift_abs": s.get("avg_trust_drift_abs"),
                "belief_total_final": s.get("belief_total_final"),
                "gossip_count": s.get("gossip_count"),
            }
        )
    for r in rows:
        print(json.dumps(r, sort_keys=True))


quick_compare("toy_dilemma_v0", ["utility_only", "routine_only", "reference"])


## Bring your own policy (adapter sketch)

1. Subclass `BaseAgent` and implement `act(...) -> (Action, ExplanationTrace)`.
2. Register a `policy_mode` in `harness/policy_registry.py` (or temporarily patch `build_policies` in a private experiment).
3. Re-run this notebook’s basic tests + `uv run pytest`.

See `ADAPTER_EXAMPLE.md` and `examples/external_stub_policy.py` in the repo root.

## Full artifact regeneration (outside the notebook)

```bash
uv run python scripts/smoke_test.py
uv run python scripts/regenerate.py
uv run python scripts/validate_outputs.py
uv run pytest
```
